# Analysis

In [1]:
import os
import pandas as pd
import numpy as np
import pyarrow.dataset as ds
import statsmodels.formula.api as smf
os.chdir("/work/Kode")
from plotnine import *
from mizani.formatters import comma_format, percent_format

os.makedirs("figures", exist_ok=True)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/work/Kode'

## Beta Estimation

Following @frazzinipedersen2014, we estimate beta as the product of a correlation component and a volatility component:

$$\hat{\beta}_i = \hat{\rho}_i \cdot \frac{\hat{\sigma}_i}{\hat{\sigma}_m}$$

The correlation $\hat{\rho}_i$ is estimated from a 5-year rolling window of monthly excess returns (minimum 3 years), while the volatility ratio is estimated from a 1-year rolling window of daily excess returns (minimum 120 observations). This two-component approach improves precision by using higher-frequency data for the noisier volatility estimate.

In [ ]:
crsp_monthly = pd.read_parquet("data/crsp_monthly.parquet")
factors_ff3_monthly = pd.read_parquet("data/factors_ff3_monthly.parquet")
factors_ff3_daily = pd.read_parquet("data/factors_ff3_daily.parquet")

crsp_daily = (
    ds.dataset("data/crsp_daily", format="parquet", partitioning="hive")
    .to_table()
    .to_pandas()
)

### Volatility Component

We estimate annualised standard deviations using a 252-day rolling window of daily excess returns (minimum 120 observations).

In [ ]:
# Merge market daily volatility
daily_vol = (
    crsp_daily
    .merge(factors_ff3_daily[["date", "mkt_excess"]], on="date", how="left")
)

# Stock volatility: 252-day rolling std of daily excess returns
stock_vol = (
    daily_vol
    .sort_values(["permno", "date"])
    .set_index("date")
    .groupby("permno")["ret_excess"]
    .rolling(window=252, min_periods=120)
    .std()
    .reset_index()
    .rename(columns={"ret_excess": "sigma_i"})
)

# Market volatility: same window on mkt_excess
market_vol = (
    daily_vol
    .drop_duplicates("date")
    .sort_values("date")
    ["mkt_excess"]
    .rolling(window=252, min_periods=120)
    .std()
    .reset_index(drop=True)
)

# Attach date back to market vol
market_vol_df = (
    daily_vol
    .drop_duplicates("date")[["date"]]
    .sort_values("date")
    .assign(sigma_m=market_vol.values)
)

vol_ratio = (
    stock_vol
    .merge(market_vol_df, on="date", how="left")
    .assign(month=lambda x: x["date"].values.astype("datetime64[M]"))
    .sort_values(["permno", "date"])
    .groupby(["permno", "month"])
    .last()
    .reset_index()
    .assign(vol_ratio=lambda x: x["sigma_i"] / x["sigma_m"])
    [["permno", "month", "vol_ratio"]]
)

### Correlation Component

We estimate the correlation with the market from a 60-month rolling window of monthly excess returns (minimum 36 months).

In [ ]:
# Pivot to wide: rows = date, cols = permno
monthly_wide = (
    crsp_monthly[["permno", "date", "ret_excess"]]
    .merge(factors_ff3_monthly[["date", "mkt_excess"]], on="date", how="left")
)

def rolling_corr(df, min_periods=36, window=60):
    df = df.sort_values("date")
    corrs = (
        df["ret_excess"]
        .rolling(window=window, min_periods=min_periods)
        .corr(df["mkt_excess"])
    )
    return corrs

corr_component = (
    monthly_wide
    .groupby("permno", group_keys=False)
    .apply(lambda df: df.assign(rho=rolling_corr(df)))
    [["permno", "date", "rho"]]
    .assign(month=lambda x: x["date"].values.astype("datetime64[M]"))
    .rename(columns={"date": "date_monthly"})
)

/tmp/ipykernel_1401/2606518727.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


### Combined Beta Estimate

The two components are combined into a time-series beta $\hat{\beta}_i^{ts} = \hat{\rho}_i \cdot (\hat{\sigma}_i / \hat{\sigma}_m)$, which is then shrunk toward the cross-sectional mean of one using Vasicek (1973) shrinkage with weights $(0.6, 0.4)$ as in @frazzinipedersen2014:

$$\hat{\beta}_i = 0.6 \cdot \hat{\beta}_i^{ts} + 0.4 \cdot 1$$

In [ ]:
beta_ts = (
    corr_component
    .merge(vol_ratio, on=["permno", "month"], how="inner")
    .assign(beta_ts=lambda x: (x["rho"] * x["vol_ratio"]).clip(lower=0.001))
    [["permno", "month", "beta_ts"]]
)

# Vasicek (1973) shrinkage toward the cross-sectional mean of 1, following F&P (2014)
# β̂ᵢ = 0.6 × β̂ᵢᵀˢ + 0.4 × 1
beta_estimates = beta_ts.assign(beta=lambda x: 0.6 * x["beta_ts"] + 0.4)[["permno", "month", "beta"]]

## BAB Portfolio Construction

### Proposition 2: The BAB Factor

Following @frazzinipedersen2014, at each month $t$ we rank stocks by their estimated beta. We form a low-beta portfolio (below-median beta) and a high-beta portfolio (above-median beta). Portfolio weights are rank-based and sum to one within each leg. The BAB factor return is:

$$r^{BAB}_t = \frac{1}{\beta^L_t} r^L_t - \frac{1}{\beta^H_t} r^H_t$$

where $\beta^L_t$ and $\beta^H_t$ are the value-weighted average betas of the low and high portfolios.

In [ ]:
# Merge betas into monthly data (beta estimated at t, applied at t+1)
crsp_monthly = crsp_monthly.assign(
    month=lambda x: x["date"].values.astype("datetime64[M]")
)

# Lag betas by one month so we use last month's beta to form this month's portfolio
beta_lagged = (
    beta_estimates
    .assign(month=lambda x: (x["month"].values.astype("datetime64[M]") + np.timedelta64(1, "M")))
)

analysis_data = (
    crsp_monthly
    .merge(beta_lagged, on=["permno", "month"], how="inner")
    .dropna(subset=["beta", "ret_excess", "mktcap_lag"])
)


def rank_weights(betas):
    """Rank-based portfolio weights (Frazzini-Pedersen eq. 9)."""
    k = betas.rank()
    k = k - k.mean()
    return k / k.abs().sum()

def bab_month(df):
    median_beta = df["beta"].median()
    low = df[df["beta"] <= median_beta].copy()
    high = df[df["beta"] > median_beta].copy()

    if low.empty or high.empty:
        return pd.Series({"r_bab": np.nan, "beta_L": np.nan, "beta_H": np.nan,
                          "r_L": np.nan, "r_H": np.nan})

    low["w"] = rank_weights(-low["beta"])   # flip so highest weight = lowest beta in low leg
    high["w"] = rank_weights(high["beta"])  # highest weight = highest beta in high leg

    beta_L = (low["w"] * low["beta"]).sum()
    beta_H = (high["w"] * high["beta"]).sum()

    r_L = (low["w"] * low["ret_excess"]).sum()
    r_H = (high["w"] * high["ret_excess"]).sum()

    r_bab = r_L / beta_L - r_H / beta_H

    return pd.Series({
        "r_bab": r_bab,
        "beta_L": beta_L,
        "beta_H": beta_H,
        "r_L": r_L,
        "r_H": r_H,
    })

bab_returns = (
    analysis_data
    .groupby("month")
    .apply(bab_month)
    .reset_index()
)

/tmp/ipykernel_1401/618596649.py:56: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [ ]:
# BAB summary statistics
bab_summary = pd.DataFrame({
    "BAB Factor": [
        bab_returns["r_bab"].mean() * 12,
        bab_returns["r_bab"].std() * np.sqrt(12),
        bab_returns["r_bab"].mean() / bab_returns["r_bab"].std() * np.sqrt(12),
        bab_returns["r_bab"].mean() / (bab_returns["r_bab"].std() / np.sqrt(bab_returns["r_bab"].notna().sum())),
    ]
}, index=["Ann. Mean Return", "Ann. Std Dev", "Sharpe Ratio", "t-statistic"])
print(bab_summary.round(4))

                  BAB Factor
Ann. Mean Return      0.0672
Ann. Std Dev          0.2566
Sharpe Ratio          0.2620
t-statistic           2.0267


In [ ]:
# BAB cumulative return
bab_cumret = (
    bab_returns
    .dropna(subset=["r_bab"])
    .assign(
        cum_bab=lambda x: (1 + x["r_bab"]).cumprod(),
        date=lambda x: x["month"].astype("datetime64[ns]")
    )
)

(
    ggplot(bab_cumret, aes(x="date", y="cum_bab"))
    + geom_line(color="steelblue", size=0.8)
    + scale_x_datetime(date_breaks="10 years", date_labels="%Y")
    + labs(x="Date", y="Cumulative Return ( Invested)")
    + theme_minimal()
).save("figures/bab_cumret.png", dpi=300, width=9, height=5)

/opt/conda/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 9 x 5 in image.
/opt/conda/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/bab_cumret.png


## Proposition 1: The Security Market Line

Proposition 1 predicts that the empirical SML is flatter than CAML implies: high-beta assets earn lower risk-adjusted returns (negative CAPM alpha) and low-beta assets earn higher risk-adjusted returns (positive CAPM alpha). We test this by running Fama-MacBeth cross-sectional regressions of excess returns on lagged betas.

In [ ]:
# Fama-MacBeth: each month regress ret_excess on beta
def fmb_month(df):
    if len(df) < 10:
        return pd.Series({"intercept": np.nan, "beta_coef": np.nan})
    m = smf.ols("ret_excess ~ beta", data=df).fit()
    return pd.Series({"intercept": m.params["Intercept"], "beta_coef": m.params["beta"]})

fmb_results = (
    analysis_data
    .groupby("month")
    .apply(fmb_month)
    .reset_index()
)

# Time-series average and Newey-West t-stats (1 lag)
cols = ["intercept", "beta_coef"]
fmb_summary = pd.DataFrame({
    "mean": fmb_results[cols].mean(),
    "t_stat": fmb_results[cols].apply(
        lambda x: x.dropna().mean() / (x.dropna().std() / np.sqrt(x.dropna().count()))
    ),
})

print(fmb_summary)

               mean    t_stat
intercept  0.007863  5.836262
beta_coef  0.001418  0.703093


/tmp/ipykernel_1401/3590990250.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


### Beta-Sorted Portfolio Alphas

We sort stocks into 10 beta-decile portfolios and compute each decile's monthly CAPM alpha to visualise the alpha-beta relationship.

In [ ]:
analysis_data = analysis_data.assign(
    beta_decile=lambda x: pd.qcut(x["beta"], q=10, labels=False) + 1
)

# Value-weighted returns per decile per month
decile_returns = (
    analysis_data
    .assign(vw_ret=lambda x: x["ret_excess"] * x["mktcap_lag"])
    .groupby(["month", "beta_decile"])
    .apply(lambda df: pd.Series({
        "ret": df["vw_ret"].sum() / df["mktcap_lag"].sum(),
        "avg_beta": (df["beta"] * df["mktcap_lag"]).sum() / df["mktcap_lag"].sum()
    }))
    .reset_index()
    .merge(
        factors_ff3_monthly[["date", "mkt_excess"]].assign(
            month=lambda x: x["date"].values.astype("datetime64[M]")
        ),
        on="month", how="left"
    )
)

# CAPM alpha per decile
def capm_alpha(df):
    m = smf.ols("ret ~ mkt_excess", data=df).fit()
    return pd.Series({
        "alpha": m.params["Intercept"],
        "alpha_t": m.tvalues["Intercept"],
        "avg_beta": df["avg_beta"].mean()
    })

decile_alphas = (
    decile_returns
    .groupby("beta_decile")
    .apply(capm_alpha)
    .reset_index()
)

print(decile_alphas)

   beta_decile     alpha   alpha_t  avg_beta
0            1  0.002741  1.746439  0.511693
1            2  0.002154  2.450884  0.676983
2            3  0.002496  3.205053  0.804396
3            4  0.001591  2.259553  0.916394
4            5  0.001141  1.801973  1.024017
5            6 -0.000145 -0.198470  1.135270
6            7  0.000748  0.748097  1.260848
7            8 -0.001768 -1.290402  1.418103
8            9 -0.004623 -2.633372  1.641384
9           10 -0.008241 -3.164249  2.103706


/tmp/ipykernel_1401/51200107.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
/tmp/ipykernel_1401/51200107.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [ ]:
# Security Market Line: empirical vs CAPM
decile_avg = (
    decile_returns
    .groupby("beta_decile")
    .agg(avg_ret=("ret", "mean"), avg_beta=("avg_beta", "mean"))
    .reset_index()
)

mkt_mean = factors_ff3_monthly["mkt_excess"].mean()
beta_range = pd.DataFrame({
    "avg_beta": np.linspace(
        decile_avg["avg_beta"].min() * 0.9,
        decile_avg["avg_beta"].max() * 1.1, 100
    )
})
beta_range["capm_ret"] = beta_range["avg_beta"] * mkt_mean

(
    ggplot(decile_avg, aes(x="avg_beta", y="avg_ret"))
    + geom_point(size=3, color="steelblue")
    + geom_smooth(method="lm", se=False, color="steelblue")
    + geom_line(
        data=beta_range,
        mapping=aes(x="avg_beta", y="capm_ret"),
        color="red", linetype="dashed"
    )
    + scale_y_continuous(labels=percent_format())
    + labs(
        x="Average Beta",
        y="Average Monthly Excess Return",
        caption="Solid blue: empirical SML. Red dashed: CAPM prediction."
    )
    + theme_minimal()
).save("figures/sml.png", dpi=300, width=7, height=5)

/opt/conda/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 7 x 5 in image.
/opt/conda/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/sml.png


In [ ]:
# CAPM alpha by beta decile
(
    ggplot(
        decile_alphas.assign(positive=lambda x: x["alpha"] > 0),
        aes(x="beta_decile", y="alpha", fill="positive")
    )
    + geom_col(show_legend=False)
    + scale_fill_manual(values={True: "#2196F3", False: "#F44336"})
    + scale_x_continuous(breaks=list(range(1, 11)))
    + scale_y_continuous(labels=percent_format())
    + labs(x="Beta Decile", y="Monthly CAPM Alpha")
    + theme_minimal()
).save("figures/decile_alphas.png", dpi=300, width=7, height=5)

/opt/conda/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 7 x 5 in image.
/opt/conda/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figures/decile_alphas.png


## Transaction Costs

We estimate the round-trip transaction cost for the BAB strategy using daily bid-ask spreads from CRSP. The proportional half-spread on day $d$ for stock $i$ is

$$c_{i,d} = \frac{\text{ask}_{i,d} - \text{bid}_{i,d}}{2 \cdot \text{mid}_{i,d}}$$

where $\text{mid}_{i,d}$ is the bid-ask midpoint. The monthly spread for each stock is the median of its daily half-spreads within the month, which is robust to days with stale or missing bid/ask quotes. We apply this cost to BAB portfolio turnover: each rebalancing incurs one half-spread on the sold leg and one on the bought leg. Following Frazzini, Israel, and Moskowitz (2015), we assume full monthly rebalancing, so this is an upper bound on realised costs.

In [ ]:
# Load daily half-spreads (computed in 03_data.qmd)
crsp_daily_all = (
    ds.dataset("data/crsp_daily", format="parquet", partitioning="hive")
    .to_table()
    .to_pandas()
    [["permno", "date", "half_spread"]]
    .dropna(subset=["half_spread"])
)

# Restrict to post-1993: CRSP bid-ask data only reliable from 1993 onwards
crsp_daily_all = crsp_daily_all[crsp_daily_all["date"] >= "1993-01-01"]

# Drop stale-quote entries: when dlybid=0, the formula gives half_spread=1.0.
# Also winsorise at 10% to remove other extreme outliers (e.g. penny stocks).
crsp_daily_all = crsp_daily_all[
    (crsp_daily_all["half_spread"] > 0) &
    (crsp_daily_all["half_spread"] <= 0.10)
]

# Monthly median of daily spreads — more robust than last() against end-of-month noise
crsp_daily_all["month"] = crsp_daily_all["date"].values.astype("datetime64[M]")
monthly_spread = (
    crsp_daily_all
    .groupby(["permno", "month"])["half_spread"]
    .median()
    .reset_index()
)

In [ ]:
# Portfolio-level spread: weight each stock's half-spread by its rank weight in each leg
def spread_month(df):
    if df.empty:
        return pd.Series({"spread_L": np.nan, "spread_H": np.nan})
    median_beta = df["beta"].median()
    low = df[df["beta"] <= median_beta].copy()
    high = df[df["beta"] > median_beta].copy()

    if low.empty or high.empty:
        return pd.Series({"spread_L": np.nan, "spread_H": np.nan})

    low["w"] = rank_weights(-low["beta"])
    high["w"] = rank_weights(high["beta"])

    spread_L = (low["w"].abs() * low["half_spread"]).sum()
    spread_H = (high["w"].abs() * high["half_spread"]).sum()
    return pd.Series({"spread_L": spread_L, "spread_H": spread_H})

spread_data = (
    analysis_data
    .merge(monthly_spread, on=["permno", "month"], how="left")
    .dropna(subset=["half_spread"])
)

portfolio_spreads = (
    spread_data
    .groupby("month")
    .apply(spread_month)
    .reset_index()
)

/tmp/ipykernel_1401/2760488451.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [ ]:
# BAB net return: subtract round-trip cost (2 × half-spread) on both legs, scaled by leverage
# r_bab_net = r_L/beta_L - r_H/beta_H - (2*spread_L)/beta_L - (2*spread_H)/beta_H
bab_tc = (
    bab_returns
    .merge(portfolio_spreads, on="month", how="left")
    .assign(
        tc_L=lambda x: 2 * x["spread_L"] / x["beta_L"].abs(),
        tc_H=lambda x: 2 * x["spread_H"] / x["beta_H"].abs(),
        r_bab_net=lambda x: x["r_bab"] - x["tc_L"] - x["tc_H"],
    )
)

# Restrict to months where TC is available so gross and net are on the same sample
bab_tc_sample = bab_tc.dropna(subset=["r_bab_net"])

def ann_mean(s):
    return s.mean() * 12

def tstat(s):
    return s.mean() / (s.std() / np.sqrt(s.notna().sum()))

# Summary: gross vs net BAB returns over the matched post-1993 sample
tc_summary = pd.DataFrame({
    "Gross BAB": [
        ann_mean(bab_tc_sample["r_bab"]),
        tstat(bab_tc_sample["r_bab"]),
    ],
    "Net BAB (after TC)": [
        ann_mean(bab_tc_sample["r_bab_net"]),
        tstat(bab_tc_sample["r_bab_net"]),
    ],
    "Avg annual TC drag": [
        ann_mean(bab_tc_sample["tc_L"] + bab_tc_sample["tc_H"]),
        np.nan,
    ],
}, index=["Ann. mean return", "t-statistic"])

print(tc_summary.round(4))

                  Gross BAB  Net BAB (after TC)  Avg annual TC drag
Ann. mean return     0.0556             -1.6157              1.6714
t-statistic          1.0664            -18.1386                 NaN
